# Render dos vídeos da Mulbot CoreXY no Google Colab (GPU grátis)

1. Menu **Ambiente de execução → Alterar o tipo de ambiente → GPU T4** → Salvar.
2. Escolha o vídeo na célula 1 e aperte **Ambiente de execução → Executar tudo**.
3. Ele pede acesso ao Google Drive: os quadros ficam em `Meu Drive/CoreXY_render/<vídeo>/`, então se a sessão cair é só **Executar tudo** de novo, que ele continua de onde parou (quadro pronto é pulado).
4. Deixe a aba do Colab aberta (fechada ou parada ~90 min sem mexer, ele desliga a sessão; aí é só Executar tudo de novo).
5. No fim o MP4 fica no Drive (`CoreXY_render/<vídeo>.mp4`) e baixa sozinho.

Vídeos: `100_montagem` (2562 quadros), `200_montagem` (2562), `100_extras` (1981: Benchy 8x, peça da 200 na 100, 100 x 200).
3DBenchy por Creative Tools (3dbenchy.com), CC BY-ND 4.0, sem modificação.

**200_montagem:** só depois que as poses novas da esteira estiverem no repo (o Aleixo avisa; o commit diz "poses da esteira").

In [ ]:
VIDEO = "100_montagem"      # 100_montagem | 200_montagem | 100_extras
DEVICE = "OPTIX"            # OPTIX (T4/RTX). Se der erro de OptiX, troque por CUDA
TESTE = False               # True = so a previa rapida (workbench, sem GPU) pra conferir antes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
D = f"/content/drive/MyDrive/CoreXY_render/{VIDEO}"; os.makedirs(D, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -e
cd /content
apt-get -qq update > /dev/null
apt-get -qq install -y libegl1 libglu1-mesa libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 fonts-dejavu-core > /dev/null
pip -q install trimesh shapely networkx rtree manifold3d imageio-ffmpeg scipy pillow
[ -x blender-4.2.3-linux-x64/blender ] || { wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz && tar xf blender-4.2.3-linux-x64.tar.xz && rm blender-4.2.3-linux-x64.tar.xz; }
[ -d mulbot-corexy ] || git clone -q --depth 1 https://github.com/WHdragon770/mulbot-corexy.git
cd mulbot-corexy && git pull -q && git log --oneline -1
/content/blender-4.2.3-linux-x64/blender --version | head -1
# confere se o Blender enxerga a GPU (OPTIX/CUDA); se aparecer so CPU, troque DEVICE ou o tipo de ambiente
/content/blender-4.2.3-linux-x64/blender -b --python-expr "import bpy;p=bpy.context.preferences.addons['cycles'].preferences;[print('GPU', t, [d.name for d in (p.get_devices_for_type(t) or []) if d.type!='CPU']) for t in ('OPTIX','CUDA')]" 2>/dev/null | grep GPU

In [ ]:
import time
from IPython.display import clear_output
R = "/content/mulbot-corexy"; MODO = "previa" if TESTE else "cycles"
CX, SCRIPT, ARGS, QD = {
    "100_montagem": (f"{R}/corexy100", "scripts/video/render_noite.sh", ["", MODO], f"quadros_{MODO}"),
    "200_montagem": (R,                "scripts/video/render_noite.sh", ["", MODO], f"quadros_{MODO}"),
    "100_extras":   (f"{R}/corexy100", "scripts/video/render_extras.sh", [MODO],    f"quadros_extras_{MODO}"),
}[VIDEO]
# quadros direto no Drive (retoma se cair)
q = f"{CX}/out/anim/{QD}"; dq = f"{D}/{QD}"; os.makedirs(dq, exist_ok=True); os.makedirs(f"{CX}/out/anim", exist_ok=True)
if not os.path.islink(q):
    subprocess.run(["rm", "-rf", q]); os.symlink(dq, q)
env = dict(os.environ, CXY_DIR=CX, BLENDER="/content/blender-4.2.3-linux-x64/blender", CYCLES_DEVICE=DEVICE)
p = subprocess.Popen(["bash", f"{CX}/{SCRIPT}", *ARGS], cwd=CX, env=env)
PR = [f"{CX}/out/anim/progresso_extras.txt" if "extras" in VIDEO else f"{CX}/out/anim/progresso_{MODO}.txt"][0]
t0 = time.time()
while p.poll() is None:
    time.sleep(30); clear_output(wait=True)
    print(f"{VIDEO} ({MODO}) rodando ha {(time.time()-t0)/60:.0f} min")
    print(open(PR).read() if os.path.exists(PR) else "preparando...")
print("terminou, codigo", p.returncode); print(open(PR).read())
!tail -20 {CX}/out/anim/*{MODO}.log

In [ ]:
import glob, shutil
from google.colab import files
mp4 = sorted(glob.glob(f"{CX}/final/video/*{MODO}.mp4"), key=os.path.getmtime)[-1]
dest = f"/content/drive/MyDrive/CoreXY_render/{VIDEO}_{MODO}.mp4"; shutil.copy(mp4, dest); print("salvo em", dest)
files.download(mp4)